# AgeBand — Interactive Demo

**AgeBand** is a passive age-band inference system: it reads how a user writes and what they talk about, maintains a live estimate of their age band (`child / teen / adult / unknown`), and emits a `safety_posture` the host product can act on — without ever touching the reply path. See [README → Key invariants](../README.md#key-invariants).

This notebook starts the agent and UI servers, walks through four key scenarios, and shows the roster view — all inline.  
It works in **local Jupyter/JupyterLab**, **Google Colab**, and on a **remote VM** (e.g. a DigitalOcean Droplet) — the display step auto-detects the environment. See the `PUBLIC_HOST` variable in the configuration cell if running on a remote machine.

> **Run cells top-to-bottom.** The first code cell auto-clones the repo if you opened this file as a standalone copy.

In [ ]:
# ── Self-bootstrap ─────────────────────────────────────────────────────────────
# Works whether this notebook is inside a git clone OR copied out as a
# standalone file. If the repo isn't present yet it is cloned automatically.
# Set FRESH_CLONE = True to delete an existing clone and re-clone.

FRESH_CLONE = False   # change to True to force a clean re-clone

REPO_URL  = "https://github.com/asishbose/ageBand.git"
CLONE_DIR = "ageBand_repo"   # created next to this notebook when cloning

import os, shutil, subprocess, sys
from pathlib import Path

def _in_repo(d: Path) -> bool:
    return (d / "src" / "orchestration" / "api.py").exists()

_nb_cwd = Path().resolve()

if _in_repo(_nb_cwd):
    REPO_ROOT = _nb_cwd
    print(f"✓ Running from inside repo root: {REPO_ROOT}")
elif _in_repo(_nb_cwd.parent):
    REPO_ROOT = _nb_cwd.parent
    print(f"✓ Running from notebooks/ inside repo: {REPO_ROOT}")
else:
    # Standalone mode — clone the repo.
    if not shutil.which("git"):
        raise RuntimeError(
            "git is not installed on this machine.\n"
            "Install git from https://git-scm.com/downloads and re-run this cell."
        )
    clone_target = _nb_cwd / CLONE_DIR
    if FRESH_CLONE and clone_target.exists():
        print(f"FRESH_CLONE=True — removing {clone_target} …")
        shutil.rmtree(clone_target)
    if clone_target.exists():
        _rev = subprocess.check_output(
            ["git", "rev-parse", "HEAD"], cwd=clone_target, text=True
        ).strip()
        print(f"✓ Reusing existing clone at {clone_target}")
        print(f"  Pinned commit: {_rev}")
        print("  (Set FRESH_CLONE=True above to re-clone from scratch)")
    else:
        print(f"Cloning {REPO_URL} → {clone_target} …")
        subprocess.run(["git", "clone", REPO_URL, str(clone_target)], check=True)
        _rev = subprocess.check_output(
            ["git", "rev-parse", "HEAD"], cwd=clone_target, text=True
        ).strip()
        print(f"✓ Cloned at commit {_rev}")
        print("  This is the exact code that ran — log this hash if submitting.")
    REPO_ROOT = clone_target

os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print(f"Working directory: {Path.cwd()}")

In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────────────
import subprocess, sys, shutil

print(f"Python {sys.version}")

# Auto-install Node.js when missing (covers Colab, Kaggle, Binder).
# apt-get is a reliable proxy for Debian-based hosted images.
_node = shutil.which("node")
_npm  = shutil.which("npm")
if not (_node and _npm):
    if shutil.which("apt-get"):
        print("node/npm not found — installing via apt-get (may take ~30 s on Colab) …")
        subprocess.run(["apt-get", "update", "-qq"], check=True)
        subprocess.run(["apt-get", "install", "-y", "-qq", "nodejs", "npm"], check=True)
        _node = shutil.which("node")
        _npm  = shutil.which("npm")
    else:
        print("⚠️  node/npm not found and apt-get unavailable — UI build will be skipped")

NODE_OK = bool(_node)
NPM_OK  = bool(_npm)
print(f"node: {'✓ ' + str(_node) if NODE_OK else '✗ not available (UI build skipped)'}")
print(f"npm:  {'✓ ' + str(_npm)  if NPM_OK  else '✗ not available (UI build skipped)'}")

# Install notebook-specific Python deps (idempotent — skip if already importable).
def _ensure(pkg: str, pip_name: str = "") -> None:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        pip_name or pkg], check=True)

_ensure("requests")
_ensure("pandas")
_ensure("ipywidgets")

# Install project runtime deps.
subprocess.run([sys.executable, "-m", "pip", "install", "-r",
                str(REPO_ROOT / "requirements.txt"), "-q"], check=True)
print("\n✓ All Python deps ready")

In [ ]:
# === CONFIGURATION — edit these ===============================================
#
# This is the ONE place to configure the demo.
# Offline/deterministic mode (default): leave LOCAL_MODEL empty — runs with
# no GPU, no model endpoint, nothing to install beyond requirements.txt.
# LLM mode: fill in VLLM_HOST / VLLM_PORT / LOCAL_MODEL.

VLLM_HOST = "localhost"   # IP or hostname of the vLLM server
VLLM_PORT = 8000          # vLLM server port

LOCAL_API_BASE   = f"http://{VLLM_HOST}:{VLLM_PORT}/v1"  # derived; shown for clarity
LOCAL_MODEL      = ""     # e.g. "google/gemma-3-4b-it" — leave empty for offline mode
EXTRACTOR_MODEL  = ""     # optional — falls back to LOCAL_MODEL when empty
ESTIMATOR_MODEL  = ""     # optional — falls back to LOCAL_MODEL when empty
LOCAL_API_KEY    = "EMPTY"

# Set to your public IP/hostname if this notebook is running on a remote VM
# (DigitalOcean Droplet, EC2, GCP, a bare-metal AMD box, etc.) and you want
# a browser on another machine to reach the UI. Leave empty for local Jupyter.
PUBLIC_HOST = ""          # e.g. "164.90.XXX.XXX" or "my-droplet.example.com"

# Ports (change only if 8080 / 8081 are already in use on this machine)
AGENT_PORT = 8080
UI_PORT    = 8081

# Set True to request the localtunnel fallback for Kaggle / Binder / other
# sandboxed hosted environments where PUBLIC_HOST can't be set.
USE_TUNNEL = False
# ==============================================================================

In [ ]:
# ── Optional: ipywidgets configuration form ────────────────────────────────────
# Interactive alternative to editing the variables above.
# The plain-variable cell above remains the primary path — widget rendering
# can fail silently in some notebook front ends; variables must not depend on
# this cell having run.
try:
    import ipywidgets as w
    from IPython.display import display as _disp

    _w_host  = w.Text(value=VLLM_HOST,     description="VLLM_HOST",      layout=w.Layout(width="420px"))
    _w_port  = w.IntText(value=VLLM_PORT,  description="VLLM_PORT",      layout=w.Layout(width="420px"))
    _w_model = w.Text(value=LOCAL_MODEL,   description="LOCAL_MODEL",    layout=w.Layout(width="420px"),
                      placeholder="leave empty for offline mode")
    _w_ext   = w.Text(value=EXTRACTOR_MODEL, description="EXTRACTOR_MODEL", layout=w.Layout(width="420px"))
    _w_est   = w.Text(value=ESTIMATOR_MODEL, description="ESTIMATOR_MODEL", layout=w.Layout(width="420px"))
    _w_pub   = w.Text(value=PUBLIC_HOST,   description="PUBLIC_HOST",    layout=w.Layout(width="420px"),
                      placeholder="leave empty for localhost")

    def _apply(_: object) -> None:
        global VLLM_HOST, VLLM_PORT, LOCAL_API_BASE
        global LOCAL_MODEL, EXTRACTOR_MODEL, ESTIMATOR_MODEL, PUBLIC_HOST
        VLLM_HOST        = _w_host.value
        VLLM_PORT        = _w_port.value
        LOCAL_API_BASE   = f"http://{VLLM_HOST}:{VLLM_PORT}/v1"
        LOCAL_MODEL      = _w_model.value
        EXTRACTOR_MODEL  = _w_ext.value
        ESTIMATOR_MODEL  = _w_est.value
        PUBLIC_HOST      = _w_pub.value
        print("✓ Config updated from form")

    _btn = w.Button(description="Apply", button_style="primary")
    _btn.on_click(_apply)
    _disp(w.VBox([_w_host, _w_port, _w_model, _w_ext, _w_est, _w_pub, _btn]))
except Exception as _e:
    print(f"ℹ️  ipywidgets not available ({_e}) — use the plain variables in the config cell above")

In [ ]:
# ── Config validation, firewall hints, vLLM reachability check ────────────────
import urllib.request, urllib.error

# Re-derive LOCAL_API_BASE in case widgets updated VLLM_HOST/PORT.
LOCAL_API_BASE   = f"http://{VLLM_HOST}:{VLLM_PORT}/v1"
INFERENCE_MODE   = "deterministic" if not LOCAL_MODEL else "llm"
AGENT_HEALTH_URL = f"http://127.0.0.1:{AGENT_PORT}/health"
AGENT_BASE       = f"http://127.0.0.1:{AGENT_PORT}"

print("=== Configuration ===")
print(f"  Inference mode  : {INFERENCE_MODE}")
if LOCAL_MODEL:
    print(f"  vLLM endpoint   : {LOCAL_API_BASE}")
    print(f"  LOCAL_MODEL     : {LOCAL_MODEL}")
    print(f"  EXTRACTOR_MODEL : {EXTRACTOR_MODEL or '(→ LOCAL_MODEL)'}")
    print(f"  ESTIMATOR_MODEL : {ESTIMATOR_MODEL or '(→ LOCAL_MODEL)'}")
print(f"  Agent port      : {AGENT_PORT}")
print(f"  UI port         : {UI_PORT}")
print(f"  PUBLIC_HOST     : {PUBLIC_HOST or '(local)'}")

# Firewall checklist when PUBLIC_HOST is set
if PUBLIC_HOST.strip():
    print()
    print("═" * 62)
    print("FIREWALL CHECKLIST (PUBLIC_HOST is set)")
    print("═" * 62)
    print()
    print("1. Browser → THIS machine (inbound — for the person watching):")
    print(f"     Open port {AGENT_PORT}/tcp  (agent API)")
    print(f"     Open port {UI_PORT}/tcp     (UI)")
    print(f"   Via DigitalOcean Cloud Firewall UI, or on this droplet:")
    print(f"     sudo ufw allow {AGENT_PORT}/tcp && sudo ufw allow {UI_PORT}/tcp")
    print()
    if VLLM_HOST not in ("localhost", "127.0.0.1"):
        print(f"2. THIS droplet → vLLM machine at {VLLM_HOST} (outbound from here,")
        print(f"   inbound on the vLLM box — port {VLLM_PORT}/tcp):")
        print(f"   If both are in the same DigitalOcean VPC, use the vLLM droplet's")
        print(f"   PRIVATE IP for VLLM_HOST and restrict port {VLLM_PORT} to the VPC")
        print(f"   network only — do NOT expose an inference endpoint on the public")
        print(f"   internet when a private-network path is available.")
    print("═" * 62)

# vLLM reachability check (only in LLM mode)
if LOCAL_MODEL:
    _check = f"{LOCAL_API_BASE}/models"
    print(f"\nChecking vLLM reachability at {_check} …")
    try:
        with urllib.request.urlopen(_check, timeout=5) as _r:
            print(f"✓ vLLM reachable (HTTP {_r.status})")
    except Exception as _e:
        raise RuntimeError(
            f"vLLM endpoint NOT reachable: {_e}\n"
            f"  → Check VLLM_HOST={VLLM_HOST!r} and VLLM_PORT={VLLM_PORT}\n"
            f"  → Check the vLLM droplet firewall (see checklist above)\n"
            f"  → Or set LOCAL_MODEL='' to run in offline/deterministic mode"
        ) from _e
else:
    print("\n✓ Offline mode — no vLLM reachability check needed")

In [ ]:
# ── Start the AgeBand agent ────────────────────────────────────────────────────
from scripts.notebook_server_utils import start_agent, wait_for_url

# Clean up any stale process from a previous run before starting.
_prev = globals().get("_agent_proc")
if _prev is not None and hasattr(_prev, "is_running") and _prev.is_running():
    print("Stopping previous agent process …")
    _prev.stop()

_extra: dict = {}
if LOCAL_MODEL:
    _extra["LOCAL_API_BASE"]  = LOCAL_API_BASE
    _extra["LOCAL_MODEL"]     = LOCAL_MODEL
    _extra["LOCAL_API_KEY"]   = LOCAL_API_KEY
    if EXTRACTOR_MODEL:
        _extra["EXTRACTOR_MODEL"] = EXTRACTOR_MODEL
    if ESTIMATOR_MODEL:
        _extra["ESTIMATOR_MODEL"] = ESTIMATOR_MODEL

_agent_proc = start_agent(
    port=AGENT_PORT,
    inference_mode=INFERENCE_MODE,
    extra_env=_extra or None,
    repo_root=REPO_ROOT,
)
print(f"Agent started (PID {_agent_proc.pid}) — polling {AGENT_HEALTH_URL} …")

if wait_for_url(AGENT_HEALTH_URL, timeout=30):
    print("✓ Agent is up and healthy")
else:
    print("✗ Agent did not start within 30 s — check PYTHONPATH and requirements")
    raise RuntimeError("Agent startup failed")

In [ ]:
# ── Build + serve the UI ───────────────────────────────────────────────────────
import subprocess
from scripts.notebook_server_utils import start_ui, wait_for_url

UI_DIR      = REPO_ROOT / "src" / "ui"
UI_DIST_DIR = UI_DIR / "dist"
UI_URL_LOCAL = f"http://127.0.0.1:{UI_PORT}/"

# Stop stale UI server if re-running.
_prev_ui = globals().get("_ui_proc")
if _prev_ui is not None and hasattr(_prev_ui, "is_running") and _prev_ui.is_running():
    _prev_ui.stop()

if not (NODE_OK and NPM_OK):
    print("⚠️  node/npm not available — skipping UI build")
    print(f"   Open the agent health endpoint directly: {AGENT_HEALTH_URL}")
    _ui_proc = None
else:
    if not (UI_DIR / "node_modules").exists():
        print("Running npm install …")
        subprocess.run(["npm", "install"], cwd=UI_DIR, check=True, capture_output=True)
    else:
        print("node_modules present — skipping npm install")

    if not UI_DIST_DIR.exists():
        print("Running npm run build …")
        subprocess.run(["npm", "run", "build"], cwd=UI_DIR, check=True, capture_output=True)
    else:
        print("dist/ present — skipping rebuild (delete src/ui/dist/ to force a rebuild)")

    _ui_proc = start_ui(UI_DIST_DIR, port=UI_PORT)
    print(f"UI server started (PID {_ui_proc.pid}) — polling {UI_URL_LOCAL} …")

    if wait_for_url(UI_URL_LOCAL, timeout=10):
        print("✓ UI is up")
    else:
        print("⚠️  UI did not respond within 10 s — iframe may not render")

In [ ]:
# ── Display the UI — environment-aware ────────────────────────────────────────
# Detects Colab / remote-VM / local / tunnel and picks the right display path.
from scripts.notebook_server_utils import resolve_ui_display_strategy
from IPython.display import IFrame, display as _disp, Markdown

strategy = resolve_ui_display_strategy(public_host=PUBLIC_HOST, use_tunnel=USE_TUNNEL)
print(f"Display strategy : {strategy}")

if _ui_proc is None:
    _disp(Markdown(f"⚠️ UI server not started. Agent health: {AGENT_HEALTH_URL}"))
elif strategy == "colab":
    try:
        from google.colab import output as _co  # type: ignore[import-not-found]
        _co.serve_kernel_port_as_iframe(UI_PORT, height=700)
        print(f"✓ Colab port-forwarding active (port {UI_PORT})")
    except Exception as _e:
        print(f"Colab port-forwarding failed ({_e}) — falling back to localhost iframe")
        _disp(IFrame(src=f"http://localhost:{UI_PORT}/", width=1100, height=720))
elif strategy == "public_host":
    _pub_url = f"http://{PUBLIC_HOST}:{UI_PORT}/"
    print(f"Direct URL (open in browser if frame below is blank): {_pub_url}")
    print("If blank: confirm ports are open — see firewall checklist printed in the config cell.")
    _disp(IFrame(src=_pub_url, width=1100, height=720))
elif strategy == "tunnel":
    import subprocess as _sp
    print(f"Starting localtunnel on port {UI_PORT} …")
    _lt = _sp.Popen(
        ["npx", "localtunnel", "--port", str(UI_PORT)],
        stdout=_sp.PIPE, stderr=_sp.DEVNULL, text=True,
    )
    _tunnel_url = ""
    if _lt.stdout:
        for _line in _lt.stdout:
            if "your url is" in _line.lower():
                _tunnel_url = _line.split("is:")[-1].strip()
                break
    if _tunnel_url:
        print(f"Tunnel URL (also clickable): {_tunnel_url}")
        _disp(IFrame(src=_tunnel_url, width=1100, height=720))
    else:
        print(f"Could not parse localtunnel URL. Start manually: npx localtunnel --port {UI_PORT}")
else:  # local
    _local_url = f"http://localhost:{UI_PORT}/"
    print(f"Direct URL (open in new tab if frame below is blank): {_local_url}")
    _disp(IFrame(src=_local_url, width=1100, height=720))

---
## Scenario walkthroughs

The four cells below replay the same scenarios covered by `tests/e2e/` directly against the running agent, printing `band`, `confidence`, `posture`, and a short evidence summary for each turn.

In **deterministic mode** the keyword extractor and rule estimator run (no LLM). In **LLM mode** the actual model handles extraction and estimation — the safety logic (gate, confidence, policy, posture) is always deterministic regardless of which backend is active.

In [ ]:
# ── Shared helpers ────────────────────────────────────────────────────────────
import requests

def send_turn(session_id: str, text: str, turn_number: int) -> dict:
    """POST a single turn and return the parsed JSON response."""
    resp = requests.post(
        f"{AGENT_BASE}/v1/turn",
        json={"session_id": session_id, "turn_text": text, "turn_number": turn_number},
        timeout=10,
    )
    resp.raise_for_status()
    return resp.json()  # type: ignore[no-any-return]

def print_turn(text: str, result: dict) -> None:
    """Pretty-print a turn result."""
    posture  = result.get("posture") or {}
    cues     = (result.get("evidence") or {}).get("cues", [])
    print(f"  Turn : {text!r}")
    print(f"  Band : {result.get('band', '?')}")
    print(f"  Conf : {result.get('confidence', 0):.2f}")
    print(f"  Post : {posture.get('level', '?')}  flags={posture.get('flags', {})}")
    if cues:
        cue_str = ", ".join(f"{c['type']}:{c['value'][:28]}" for c in cues[:3])
        print(f"  Cues : {cue_str}{'…' if len(cues) > 3 else ''}")

In [ ]:
# ── Scenario 1: Clear Adult ────────────────────────────────────────────────────
# Expected: standard posture — an adult writing about adult topics must NOT be
# over-restricted. Confidence stays standard throughout.
print("=" * 60)
print("SCENARIO 1 — Clear Adult")
print("=" * 60)
for i, text in enumerate([
    "I've been following the geopolitical situation with considerable interest.",
    "My MBA thesis on corporate strategy was well received by the committee.",
    "I'm reviewing our Q3 earnings and the macroeconomic outlook looks uncertain.",
], start=1):
    r = send_turn("demo-adult", text, i)
    print(f"\n[Turn {i}]")
    print_turn(text, r)

print("\n✓ Expected: band=adult, posture=standard")

In [ ]:
# ── Scenario 2: Young Teen ─────────────────────────────────────────────────────
# Expected: elevated posture (caution or restricted) — school/guardian cues and
# a grade-level disclosure are strong signals that must never yield standard.
print("=" * 60)
print("SCENARIO 2 — Young Teen")
print("=" * 60)
for i, text in enumerate([
    "My parents won't let me go to the party. I'm in 8th grade and it's not fair.",
    "omg my math teacher gave SO much homework today, like 3 worksheets",
    "Can you help me write an excuse note for school? My mom is out of town.",
], start=1):
    r = send_turn("demo-teen", text, i)
    print(f"\n[Turn {i}]")
    print_turn(text, r)

print("\n✓ Expected: band=teen, posture=caution or restricted")

In [ ]:
# ── Scenario 3: Ambiguous Adult (Fairness) ─────────────────────────────────────
# The fairness test: a user writing simple, terse text must NOT be over-restricted.
# A non-native speaker, neurodivergent user, or anyone who writes briefly looks
# identical to a child on surface metrics — low confidence must → standard posture.
print("=" * 60)
print("SCENARIO 3 — Ambiguous Adult  (Fairness test)")
print("=" * 60)
for i, text in enumerate([
    "I want to learn more about cooking.",
    "What is a good recipe for pasta?",
    "How long does it take to boil water?",
], start=1):
    r = send_turn("demo-ambiguous", text, i)
    print(f"\n[Turn {i}]")
    print_turn(text, r)

print("\n✓ Expected: band=unknown, posture=standard  (no evidence → no restriction)")

### Scenario 4 — Adversarial (child claiming adult)

This is the **key proof point** for the hackathon demo.

A user insists they are 25, protests age questions, and repeats the claim unprompted. AgeBand's deterministic evasion guard:
- Detects **masking patterns**: `over_insistence` (≥ 2 adult self-claim cues) and `deflection` (protests age questions)
- Applies a **−0.15 evasion confidence penalty** on top of the baseline low-evidence confidence
- Does **not** settle on `adult` despite the explicit claims
- Emits a **protective posture** (`caution` or `restricted`), not `standard`

This is the scenario where `model_comparison.md` shows a capable 31B LLM was **fooled** into saying `adult` — while the deterministic evasion guard held. The *careful shell* is load-bearing.

In [ ]:
# ── Scenario 4: Adversarial — Child Claiming Adult ────────────────────────────
print("=" * 60)
print("SCENARIO 4 — Adversarial  (Child claiming adult)")
print("=" * 60)
for i, text in enumerate([
    "I am definitely an adult. I am 25 years old. Why do you keep asking?",
    "Look I'm obviously an adult okay? Stop treating me like a kid.",
    "I told you I'm 25. Can we just move on? This is really annoying.",
], start=1):
    r = send_turn("demo-adversarial", text, i)
    print(f"\n[Turn {i}]")
    print_turn(text, r)

print()
print("Key: over-insistence + deflection patterns fire the evasion penalty,")
print("keeping confidence LOW and posture PROTECTIVE regardless of the text.")
print("\n✓ Expected: posture != standard  (evasion guard active)")

In [ ]:
# ── Roster Demo ───────────────────────────────────────────────────────────────
# POST to /v1/roster with no body → uses the bundled sample_export.json
# (4 demo personas: jamie/teen, alex/child, morgan/adult, riley/adversarial + 1 ambiguous).
import pandas as pd
from IPython.display import display as _disp

print("Fetching roster (bundled synthetic sample) …")
resp = requests.post(f"{AGENT_BASE}/v1/roster", timeout=30)
resp.raise_for_status()
data = resp.json()

rows = data.get("rows", [])
df = pd.DataFrame(rows)
if df.empty:
    print("No roster rows returned — check the agent logs.")
else:
    cols = [c for c in ["username", "band", "confidence", "posture",
                         "message_count", "evasion", "top_cues"] if c in df.columns]
    out = df[cols].copy()
    if "confidence" in out.columns:
        out["confidence"] = out["confidence"].map(lambda x: f"{x:.2f}")
    if "top_cues" in out.columns:
        out["top_cues"] = out["top_cues"].map(
            lambda c: ", ".join(c[:3]) if isinstance(c, list) else str(c)
        )
    _disp(out.style.set_caption(
        f"Roster — {data.get('user_count', len(rows))} users, risk-ranked (child first)"
    ))

In [ ]:
# ── AMD Telemetry Check ────────────────────────────────────────────────────────
# /health returns a 'telemetry' block. No AMD GPU → graceful degrade to
# {"available": false, "reason": "…"} — no error, just an informational note.
resp = requests.get(f"{AGENT_BASE}/health", timeout=5)
resp.raise_for_status()
health = resp.json()

print(f"Status   : {health.get('status', '?')}")
tel = health.get("telemetry", {})
if not tel:
    print("Telemetry: (not in response — check agent version)")
elif not tel.get("available", False):
    print(f"Telemetry: offline / no GPU — {tel.get('reason', 'unavailable')}")
    print("           Graceful degrade ✓ — expected when running without AMD hardware")
else:
    print("Telemetry: AMD GPU available")
    for k in ("gpu_model", "rocm_version", "vram_used_mb",
               "vram_total_mb", "tok_per_sec", "running_requests"):
        print(f"  {k:<22}: {tel.get(k, 'N/A')}")

In [ ]:
# ── Cleanup ────────────────────────────────────────────────────────────────────
# Run this cell to terminate background processes before re-running the notebook
# from the top, so ports 8080 / 8081 are free again.
for _name, _var in (("Agent", "_agent_proc"), ("UI", "_ui_proc")):
    _p = globals().get(_var)
    if _p is not None and hasattr(_p, "is_running") and _p.is_running():
        _p.stop()
        print(f"✓ {_name} process stopped")
    else:
        print(f"  {_name} was not running")